# Class probability

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

In [ ]:
X = pd.read_csv("../../_temp/v1/X.csv", index_col=[0, 1, 2])
Xpred = pd.read_csv("../../_temp/v1/Xpred_2D.csv", index_col=[0, 1, 2])
Delaunay = pd.read_csv("../../_temp/v1/delaunay.Xpred_2D.csv", index_col=[0, 1, 2])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

## Plot

In [ ]:
N_COLORS = 8
probability_levels = np.linspace(0, 1, N_COLORS + 1)
probability_norm = mcolors.BoundaryNorm(probability_levels, ncolors=N_COLORS)
phi_1_cmap = mcolors.LinearSegmentedColormap.from_list(
    "phi_1_blue", ["white", "tab:blue"], N=N_COLORS
)
phi_3_cmap = mcolors.LinearSegmentedColormap.from_list(
    "phi_3_red", ["white", "tab:red"], N=N_COLORS
)

### Marginal class probability (phi_1)

In [ ]:
marginal = pd.read_csv(
    "../../benchmarks/v1/gpqr.phi_1.class_marginal.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)
marginal = marginal[marginal["target"] == "phi_1"].drop(columns=["target"])
marginal = marginal.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=phi_1_cmap,
        norm=probability_norm,
        levels=probability_levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=phi_1_cmap, norm=probability_norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("P(φ₁ < 0)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

### Marginal class probability (phi_3)

In [ ]:
marginal = pd.read_csv(
    "../../benchmarks/v1/gpqr.phi_3.class_marginal.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)
marginal = marginal[marginal["target"] == "phi_3"].drop(columns=["target"])
marginal = marginal.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=phi_3_cmap,
        norm=probability_norm,
        levels=probability_levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=phi_3_cmap, norm=probability_norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("P(φ₃ < 0)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## phi_1 vs phi_3

In [ ]:
# Blue: P(phi_1 < 0) dominates. Red: P(phi_3 < 0) dominates.
def load_marginal_probability(path, target):
    probability = pd.read_csv(path, index_col=["index", "batch", "sample"])
    probability = probability[probability["target"] == target].drop(columns=["target"])
    return probability.groupby(level="index").mean()["marginal_prob"]


prob_type_1 = load_marginal_probability(
    "../../benchmarks/v1/gpqr.phi_1.class_marginal.Xpred_2D.csv", "phi_1"
)
prob_type_2 = load_marginal_probability(
    "../../benchmarks/v1/gpqr.phi_3.class_marginal.Xpred_2D.csv", "phi_3"
)
class_prob = pd.concat(
    [prob_type_1.rename("type_1"), prob_type_2.rename("type_2")], axis=1
).fillna(0.0)

p1 = class_prob["type_1"].to_numpy()
p2 = class_prob["type_2"].to_numpy()
probability_difference = p2 - p1
comparison_levels = np.linspace(-1, 1, 17)
comparison_norm = mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
comparison_cmap = mcolors.LinearSegmentedColormap.from_list(
    "phi_1_blue_phi_3_red", ["tab:blue", "white", "tab:red"]
)

fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    shape = this_Xpred.shape[1:-1]
    x = this_Xpred[0, ...].squeeze(axis=-1)
    y = this_Xpred[1, ...].squeeze(axis=-1)

    contour = ax.contourf(
        x,
        y,
        probability_difference[ok_pred.to_numpy()].reshape(shape),
        levels=comparison_levels,
        cmap=comparison_cmap,
        norm=comparison_norm,
        extend="both",
    )

    delaunay = Delaunay[ok_pred.to_numpy()].to_numpy().reshape(shape)
    ax.contour(x, y, delaunay.astype(float), levels=[0.5], colors="k")
    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(contour, cax=cbar_ax, orientation="horizontal")
cbar.set_ticks([-1, 0, 1])
cbar.set_ticklabels(["P(φ₁ < 0) dominates", "Equal probability", "P(φ₃ < 0) dominates"])
cbar.set_label("P(φ₃ < 0) − P(φ₁ < 0)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()